In [36]:
# Import des bibliothèques

import os
from pathlib import Path
import numpy as np
import pandas as pd
import sys
import time
import gc
import random as rd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from tokenizers import Tokenizer
# from transformers import AutoTokenizer
from transformers import BertForMaskedLM, AutoTokenizer, AutoModelForMaskedLM
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace


In [37]:
# Détermination du Path

DATASET_PATH = Path("../data/text")
print(DATASET_PATH)


..\data\text


In [38]:
# Récupération des données textuelles

VOC_SIZE = 1000

def load_data(datapath, max_size=None):
    texts_files = list(datapath.glob("*.txt"))
    texts = []  
    for files in texts_files:
        with open(files, "r", encoding='utf8') as files:
            text = files.readlines()
            texts += text
    texts = list(set(texts))
    
    return texts

texts = load_data(DATASET_PATH)

In [39]:
# Chargement du modèle CamemBERTav2 et tokenisation du texte

model_checkpoint = "almanach/camembertav2-base"

tokenizerCamemBERTaV2_FT = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)
modelCamemBERTaV2_FT = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

inputs = tokenizerCamemBERTaV2_FT(texts, return_tensors='pt', max_length=100, 
                   truncation=True, padding='max_length')

inputs['labels'] = inputs.input_ids.detach().clone()

print(inputs.tokens(1))


Some weights of DebertaV2ForMaskedLM were not initialized from the model checkpoint at almanach/camembertav2-base and are newly initialized: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


['[CLS]', 'Oil', '##s', 'and', 'fat', '##s', 'Cru', '##de', 'pal', '##m', 'o', '##il', 'Cru', '##de', 'o', '##il', 'extra', '##cted', 'from', 'the', 'pul', '##p', 'of', 'the', 'fruit', 'of', 'the', 'o', '##il', 'pal', '##m', 'tre', '##e', '(', 'us', '##ual', '##ly', 'El', '##ae', '##is', 'gui', '##ne', '##ensis', 'Jac', '##q', '.).', '\n', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [40]:
tokenizerCamemBERTaV2_FT.special_tokens_map

{'bos_token': '[CLS]',
 'eos_token': '[SEP]',
 'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]'}

In [41]:
CLS_id = tokenizerCamemBERTaV2_FT.encode('[CLS]')[1]
MASK_id = tokenizerCamemBERTaV2_FT.encode('[MASK]')[1]
PAD_id = tokenizerCamemBERTaV2_FT.encode('[PAD]')[1]
SEP_id = tokenizerCamemBERTaV2_FT.encode('[SEP]')[1]
print(f'PAD token id : {PAD_id}')
print(f'MASK token id : {MASK_id}')
print(f'CLS token id : {CLS_id}')
print(f'SEP token id : {SEP_id}')


PAD token id : 0
MASK token id : 4
CLS token id : 1
SEP token id : 2


In [42]:
# Préparation des données pour le Modèle de language maskey

rand = torch.rand(inputs.input_ids.shape)
mask_arr = (rand < 0.15) * (inputs.input_ids != CLS_id) * (inputs.input_ids != PAD_id) * (inputs.input_ids != SEP_id)

inputs.input_ids[mask_arr] = MASK_id

sample_idx = [i for i in range(len(inputs.input_ids))]

shuffled_sample_idx = rd.sample(sample_idx, len(sample_idx))

train_idx = shuffled_sample_idx[:int(0.70*len(shuffled_sample_idx))]
val_idx = shuffled_sample_idx[int(0.70*len(shuffled_sample_idx)):int(0.85*len(shuffled_sample_idx))]
test_idx = shuffled_sample_idx[int(0.85*len(shuffled_sample_idx)):]

In [43]:
# Préparation du dataset

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, idx):
        self.encodings = encodings
        self.idx = idx
        self.encodings = {key: [val[i] for i in self.idx] for key, val in self.encodings.items()}
        
    def __getitem__(self, idx):
        return {key : torch.tensor(val[idx]) for key, val in self.encodings.items()}
    
    def __len__(self):
        return len(self.encodings['input_ids'])

dataset_train = CustomDataset(inputs, train_idx)
dataset_val = CustomDataset(inputs, val_idx)
dataset_test = CustomDataset(inputs, test_idx)

train_dataloaded = torch.utils.data.DataLoader(dataset_train, batch_size=16, shuffle=True)
val_dataloaded = torch.utils.data.DataLoader(dataset_val, batch_size=16, shuffle=True)
test_dataloaded = torch.utils.data.DataLoader(dataset_test, batch_size=16, shuffle=True)

In [44]:
#class MLM_model(nn.Module):
#    def __init__(self, model):
#        super(MLM_model, self).__init__()
#        self.history = {"epochs":[], "test":[]}
#        self.model = model
    
#    def parameters(self):
#        return self.model.parameters()

#    def forward(self, x, attention_mask, labels):
#        return self.model(x, attention_mask, labels)
    
#    def train_log(self, train_batch_losses, val_batch_losses, train_loss, validation_loss):
#        self.history["epochs"].append({"train_batch_losses":train_batch_losses, 
#                                "val_batch_losses":val_batch_losses, 
#                                "train_loss":train_loss, 
#                                "validation_loss":validation_loss})
    
#    def test_log(self, test_batch_losses, test_loss):
#        self.history["test"].append({"test_batch_losses":test_batch_losses,
#                                "test_loss":test_loss})

In [45]:
# Définition du device 

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
#model = MLM_model(model)
modelCamemBERTaV2_FT.to(device)
print(device)

cpu


In [46]:
# Apprentissage

def train_step(module, batch, batch_idx, optimizer):
    module.train(True)
    
    inputs_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)
    
    outputs = module(inputs_ids, attention_mask, labels=labels)
    
    loss = outputs.loss
    print(f"\n\033[1;37mBatch loss {batch_idx+1} : {loss.item()}")
    loss.backward()
    
    torch.nn.utils.clip_grad_norm_(module.parameters(), max_norm=1.0)
    optimizer.step()
    optimizer.zero_grad()
    
    return module, loss

def eval_step(module, batch, batch_idx, optimizer=None, training=True):
    if training == False :
            module.to('cpu')
            
    with torch.no_grad():
            
        inputs_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
    
        outputs = module(inputs_ids, attention_mask, labels=labels)
    
        loss = outputs.loss
         
        if training:
            print(f"\n\033[1;32mValidation Batch loss {batch_idx+1} : {loss.item()}")
            return module, loss
        else:
            print(f"\n\033[1;32mTest Batch loss {batch_idx+1} : {loss.item()}")
            return module, loss, outputs, labels

def train_loop(module, EPOCHS, train_dataset, val_dataset, optimizer, lr_scheduler=None):
    for epoch in range(EPOCHS):
        deb=time.time()
        
        module.train(True)
        
        train_batch_losses = []
        for batch_idx in range(len(train_dataset)):
            batch = next(iter(train_dataset))
            module, loss = train_step(module, batch, batch_idx, optimizer)
            train_batch_losses.append(loss.item())
            
        if lr_scheduler is not None:
          lr_scheduler.step()
        train_loss = np.mean(train_batch_losses)

        module.train(False)
        val_batch_losses = []
        for batch_idx in range(len(val_dataset)):
            batch = next(iter(val_dataset))
            module, loss = eval_step(module, batch, batch_idx)
            val_batch_losses.append(loss.item())
        val_loss = np.mean(val_batch_losses)

#        module.train_log(train_batch_losses, val_batch_losses, train_loss, val_loss)
        print(f"\n\033[1;33mEpoch {epoch+1} :\n\033[1;37mTraining Loss : {train_loss}")
        print(f"\033[1;32mValidation Loss : {val_loss}")
        print(f"\033[1;31mDurée epoch : {time.time()-deb} secondes")
    return module

def evaluate(module, test_dataset):
    module.train(False)
    test_batch_losses = []
    predictions = []
    true_targets = []
    for batch_idx in range(len(test_dataset)):
        batch = next(iter(test_dataset))
        module, loss, outputs, labels = eval_step(module, batch, batch_idx, training=False)

        test_batch_losses.append(loss.item())
        predictions.append(outputs)
        true_targets.append(labels)

    test_loss = np.mean(test_batch_losses)
#    module.test_log(test_batch_losses, test_loss)
    print(f"\nTest Loss : {test_loss}")
    return predictions, true_targets

In [47]:
# Entrainement

if __name__ == "__main__":
    EPOCHS = 1
    LR = 1e-4
    
    optimizer = torch.optim.Adam(modelCamemBERTaV2_FT.parameters(), lr=LR, eps=5e-8)
    module = train_loop(module=modelCamemBERTaV2_FT,
                        EPOCHS=EPOCHS,
                        train_dataset=train_dataloaded, 
                        val_dataset=val_dataloaded,
                        optimizer=optimizer)
    device = 'cpu'
    predictions, true_targets = evaluate(module, 
                                         test_dataloaded)



C:\Users\rapha\AppData\Local\Temp\ipykernel_59344\3187914882.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return {key : torch.tensor(val[idx]) for key, val in self.encodings.items()}



Batch loss 1 : 16.06073760986328


KeyboardInterrupt: 

In [ ]:
print(torch.cuda.memory_stats())

OrderedDict({'active.all.allocated': 1895106, 'active.all.current': 408, 'active.all.freed': 1894698, 'active.all.peak': 1028, 'active.large_pool.allocated': 1189601, 'active.large_pool.current': 152, 'active.large_pool.freed': 1189449, 'active.large_pool.peak': 489, 'active.small_pool.allocated': 705505, 'active.small_pool.current': 256, 'active.small_pool.freed': 705249, 'active.small_pool.peak': 725, 'active_bytes.all.allocated': 9850197817344, 'active_bytes.all.current': 910587904, 'active_bytes.all.freed': 9849287229440, 'active_bytes.all.peak': 5074602496, 'active_bytes.large_pool.allocated': 9740387545088, 'active_bytes.large_pool.current': 909324288, 'active_bytes.large_pool.freed': 9739478220800, 'active_bytes.large_pool.peak': 5065654272, 'active_bytes.small_pool.allocated': 109810272256, 'active_bytes.small_pool.current': 1263616, 'active_bytes.small_pool.freed': 109809008640, 'active_bytes.small_pool.peak': 10062336, 'allocated_bytes.all.allocated': 9850197817344, 'allocate

In [ ]:
print(inputs.input_ids.max())
print(inputs.input_ids.min())

tensor(32348)
tensor(0)


In [ ]:
# enregistrement du modèle

modelCamemBERTaV2_FT.save_pretrained('./saves/model/CamemBERTaV2_FT')
tokenizerCamemBERTaV2_FT.save_pretrained('./saves/tokenizer/CamemBERTaV2_FT')

('./saves/tokenizer/CamemBERTaV2_FT\\tokenizer_config.json',
 './saves/tokenizer/CamemBERTaV2_FT\\special_tokens_map.json',
 './saves/tokenizer/CamemBERTaV2_FT\\vocab.txt',
 './saves/tokenizer/CamemBERTaV2_FT\\added_tokens.json',
 './saves/tokenizer/CamemBERTaV2_FT\\tokenizer.json')